# Cell Cycle Classification

## Imports

In [1]:
%matplotlib inline

In [ ]:
from __future__ import print_function
import h5py
import sys
import os
from os.path import exists
sys.path.append('../')
sys.path.append('../core')
import scipy as SP
from cyclone import cyclone
from load_data import *
import joblib

CFG = {}
CFG['test_file'] = '../../data/data_mESCbulk.h5f'
#CFG['test_file'] = './data/normCounts_mESCquartz.h5f'
CFG['train_file'] = '../../data/normCountsBuettnerEtAl.h5f'
CFG['util_file'] = '../../data/normCounts_mESCquartz.h5f'

out_dir = './out_mESCbulk/'

## Data loading

In [ ]:
data = load_data(CFG, is_Ens=True, het_only=True, het_onlyCB=False, gene_set='GOCB')#gene_set can be either a list of genes,
class_labels = data['class_labels']#['G1','G2M','S']#['T-cells']#d#['Liver']#['early', 'late', 'mid']#data['class_labels']#['G1', 'S','G2M']#['Liver']#[data['class_labels']#['T-cells']##['G1', 'S','G2M']#['T-cells']#['G1', 'S','G2M']# #                                                                                        #or 'all' (all genes), 'GOCB' GO and cyclebase or 'CB' or 'GO'
data.keys()

print(data['cc_ens'].shape[0], 'Cell cycle genes used for training and prediction')
print(data['class_labels'])

## Model training

In [ ]:
cyclone = cyclone(data['Y'],row_namesY= data['genes'],cc_geneNames= data['cc_ens'],labels = data['labels'], 
                         Y_tst = data['Y_test'], row_namesY_tst = data['genes_tst'], labels_tst = data['labels_tst'])

In [ ]:
cyclone.trainModel(rftop = 40, cv=2, out_dir = out_dir, do_pca=1, npc=1, is_SVM=0)

### Model exporting/loading

In [ ]:
# after trainModel()
joblib.dump(cyclone, 'cyclone_model.joblib')

In [ ]:
# later
cyclone = joblib.load('cyclone_model.joblib')

In [ ]:
# reuse stored predictions
pred = cyclone.scores_tst.argmax(axis=1) + 1   # RF on test set

# replot
cyclone.plotHistograms(out_dir='./out', method='RF')
cyclo

In [ ]:
ne.plotScatter(out_dir='./out', method='RF')

## Training analysis

In [ ]:
cyclone.plotHistograms(class_labels = class_labels, out_dir = out_dir, method='GNB', do_h=True)
cyclone.plotPerformance(plot_test=False, out_dir =out_dir, method='GNB')

In [ ]:
cyclone.plotScatter(plot_test = True, xaxis = 0, yaxis = 2, xlab = 'G1 score', ylab = 'G2M score', class_labels = class_labels, out_dir = out_dir, method='GNB')
cyclone.plotScatter(plot_test = False, xaxis = 0, yaxis = 2, xlab = 'G1 score', ylab = 'G2M score', class_labels = ['G1', 'S', 'G2M'], out_dir = out_dir, method='GNB')